# EV Spare Parts Inventory & Forecasting Analysis

## Project overview
We supply spare parts to several EV service centers from a central warehouse. Using weekly
historical demand and inventory data, this notebook investigates demand patterns, inventory
performance, and forecasting accuracy, then connects forecast quality to real business cost.

## Central question
**Can a forecasting method with low statistical error still create high inventory cost?**

A method can look good on paper (low MSE/MAE/MAPE) while still being expensive in practice,
if its errors are systematically biased in one direction (e.g. consistently under-forecasting
a part with a high stockout cost, or over-forecasting a part with a high holding cost). Every
section below builds toward answering this question, comparing methods both on statistical
error and on the business cost they would have generated.

## Data
Three weekly CSVs live in `data/`, joined on (`week_start`, `part_id`):
- **parts.csv** -- one row per part: identity, cost, price, supplier, lead time, cost
  parameters used later for reorder logic.
- **demand.csv** -- weekly units requested per part, by service center region and vehicle type.
- **inventory.csv** -- weekly stock on hand and replenishment, recorded *before* that week's
  demand is fulfilled.

## Analysis plan
1. **Data audit** (this section) -- shapes, dtypes, missing values, duplicates, date gaps,
   negative values.
2. Merge + derived variables, with a sanity check that demand reconciles with fulfilled + unmet.
3. Exploratory analysis of demand, stockouts, fill rate, and cost exposure.
4. Forecasting with several **candidate** methods (naive, moving average, exponential
   smoothing, Holt-Winters, Croston's for intermittent demand) -- each included only if the
   data actually justifies it; Phase 3's diagnostics decide which candidates survive.
5. Dual evaluation: statistical error vs. business cost, per method.
6. Reorder logic and the final `reorder_recommendations.csv` output.


## Phase 1: Data audit

Before doing any analysis we need to know whether the data can be trusted at face value.
For each of the three source files we check:
- **Shape and dtypes** — do columns have the types we expect (dates as dates, IDs as strings,
  quantities as numbers)?
- **Missing values** — are there gaps that would break a merge or a calculation?
- **Duplicates** — is `(part_id, week_start)` unique in `demand` and `inventory`, and is
  `part_id` unique in `parts`? A duplicate key would silently corrupt the merge in Phase 2.
- **Date gaps** — does every part have a reading for every week in the observation window?
- **Negative values** — quantities, costs, and lead times should never be negative.


In [63]:
import pandas as pd
import numpy as np

# Load the three source tables. week_start is parsed as a date up front since
# every downstream step (merging, gap-checking, forecasting) depends on it.
parts = pd.read_csv("../data/parts.csv")
demand = pd.read_csv("../data/demand.csv", parse_dates=["week_start"])
inventory = pd.read_csv("../data/inventory.csv", parse_dates=["week_start"])

print("parts:", parts.shape)
print("demand:", demand.shape)
print("inventory:", inventory.shape)

parts.head()


parts: (24, 10)
demand: (1872, 6)
inventory: (1872, 4)


,part_id,part_name,category,unit_cost,unit_price,supplier,lead_time_days,demand_trend,holding_cost_per_unit_week,stockout_cost_per_unit
0,P001,Battery Module A,Battery System,820,1150,VoltCore GmbH,21,Increasing,4.92,847.50
1,P002,Battery Cooling Plate,Battery System,145,240,ThermoEV Supply,14,Increasing,0.87,203.00
2,P003,Battery Management Sensor,Battery System,95,165,SensorTech EU,14,Stable,0.57,144.25
3,P004,Charging Port Assembly,Charging System,210,340,ChargeLink AG,21,Increasing,1.26,249.00
4,P005,AC Charging Cable,Cables and Connectors,75,135,CableWorks,7,Stable,0.45,93.75


In [64]:
# Check dtypes: week_start should be datetime64, IDs/categoricals should be
# object (string), and quantities/costs should be int64 or float64.
print("=== parts dtypes ===")
print(parts.dtypes)
print("\n=== demand dtypes ===")
print(demand.dtypes)
print("\n=== inventory dtypes ===")
print(inventory.dtypes)


=== parts dtypes ===
part_id                        object
part_name                      object
category                       object
unit_cost                       int64
unit_price                      int64
supplier                       object
lead_time_days                  int64
demand_trend                   object
holding_cost_per_unit_week    float64
stockout_cost_per_unit        float64
dtype: object

=== demand dtypes ===
week_start               datetime64[ns]
part_id                          object
units_demanded                    int64
service_center_region            object
vehicle_type                     object
service_campaign                  int64
dtype: object

=== inventory dtypes ===
week_start       datetime64[ns]
part_id                  object
stock_on_hand             int64
replenishment             int64
dtype: object


In [65]:
# Check for missing values in each table.
print("=== parts missing values ===")
print(parts.isna().sum())
print("\n=== demand missing values ===")
print(demand.isna().sum())
print("\n=== inventory missing values ===")
print(inventory.isna().sum())


=== parts missing values ===
part_id                       0
part_name                     0
category                      0
unit_cost                     0
unit_price                    0
supplier                      0
lead_time_days                0
demand_trend                  0
holding_cost_per_unit_week    0
stockout_cost_per_unit        0
dtype: int64

=== demand missing values ===
week_start               0
part_id                  0
units_demanded           0
service_center_region    0
vehicle_type             0
service_campaign         0
dtype: int64

=== inventory missing values ===
week_start       0
part_id          0
stock_on_hand    0
replenishment    0
dtype: int64


In [66]:
# Check for duplicate keys: part_id must be unique in parts.csv, and
# (part_id, week_start) must be unique in demand and inventory.
print("duplicate part_id in parts:", parts.duplicated(subset=["part_id"]).sum())
print("duplicate (part_id, week_start) in demand:", demand.duplicated(subset=["part_id", "week_start"]).sum())
print("duplicate (part_id, week_start) in inventory:", inventory.duplicated(subset=["part_id", "week_start"]).sum())

# Also confirm the same set of parts appears in all three tables.
parts_ids = set(parts["part_id"])
demand_ids = set(demand["part_id"])
inventory_ids = set(inventory["part_id"])
print("\nparts not in demand:", parts_ids - demand_ids)
print("demand parts not in parts.csv:", demand_ids - parts_ids)
print("inventory parts not in parts.csv:", inventory_ids - parts_ids)


duplicate part_id in parts: 0
duplicate (part_id, week_start) in demand: 0
duplicate (part_id, week_start) in inventory: 0

parts not in demand: set()
demand parts not in parts.csv: set()
inventory parts not in parts.csv: set()


In [67]:
# Unique part counts and the observation window (date range) for each table.
print("unique part_ids:", parts["part_id"].nunique())
print("\ndemand date range:", demand["week_start"].min(), "to", demand["week_start"].max())
print("inventory date range:", inventory["week_start"].min(), "to", inventory["week_start"].max())

n_weeks = demand["week_start"].nunique()
print("\nunique weeks in demand:", n_weeks)
print("parts x weeks:", parts["part_id"].nunique() * n_weeks, "vs demand rows:", len(demand))


unique part_ids: 24

demand date range: 2024-01-01 00:00:00 to 2025-06-23 00:00:00
inventory date range: 2024-01-01 00:00:00 to 2025-06-23 00:00:00

unique weeks in demand: 78
parts x weeks: 1872 vs demand rows: 1872


In [72]:
# Explicit per-part date-gap check: build the full expected set of weeks,
# then compare it against each part's actual weeks in demand and inventory.
expected_weeks = pd.date_range(demand["week_start"].min(), demand["week_start"].max(), freq="7D")
def find_gaps(df, name):
    gaps = {}
    for pid, group in df.groupby("part_id"):
        missing = set(expected_weeks) - set(group["week_start"])
        if missing:
            gaps[pid] = sorted(missing)
    if gaps:
        print(f"{name}: parts with missing weeks -> {gaps}")
    else:
        print(f"{name}: no missing weeks for any part ({len(expected_weeks)} expected weeks each)")

find_gaps(demand, "demand")
find_gaps(inventory, "inventory")


demand: no missing weeks for any part (78 expected weeks each)
inventory: no missing weeks for any part (78 expected weeks each)


In [73]:
# Negative-value check: none of these quantities/costs should ever be negative.
numeric_checks = {
    "parts.unit_cost": parts["unit_cost"],
    "parts.unit_price": parts["unit_price"],
    "parts.lead_time_days": parts["lead_time_days"],
    "parts.holding_cost_per_unit_week": parts["holding_cost_per_unit_week"],
    "parts.stockout_cost_per_unit": parts["stockout_cost_per_unit"],
    "demand.units_demanded": demand["units_demanded"],
    "inventory.stock_on_hand": inventory["stock_on_hand"],
    "inventory.replenishment": inventory["replenishment"],
}

for label, series in numeric_checks.items():
    n_negative = (series < 0).sum()
    print(f"{label}: {n_negative} negative values")


parts.unit_cost: 0 negative values
parts.unit_price: 0 negative values
parts.lead_time_days: 0 negative values
parts.holding_cost_per_unit_week: 0 negative values
parts.stockout_cost_per_unit: 0 negative values
demand.units_demanded: 0 negative values
inventory.stock_on_hand: 0 negative values
inventory.replenishment: 0 negative values


### Phase 1 summary — GO

The audit found no issues:
- **Shape**: 24 parts, 1872 weekly rows each in `demand` and `inventory` (24 parts x 78 weeks,
  exactly as expected — a complete rectangular panel).
- **Dtypes**: all columns typed as expected (`week_start` as datetime, IDs/categories as
  strings, quantities/costs as int/float). No coercion needed.
- **Missing values**: none, in any of the three tables.
- **Duplicate keys**: `part_id` is unique in `parts`; `(part_id, week_start)` is unique in
  both `demand` and `inventory`. All three tables reference the same 24 part_ids — no orphans
  on either side. No duplicate-resolution step is needed before merging.
- **Date gaps**: every part has all 78 weeks (2024-01-01 to 2025-06-23) in both `demand` and
  `inventory` — no missing weeks to worry about being misread as zero-demand.
- **Negative values**: none found across any cost, quantity, or lead-time column.

The data is clean enough to merge as-is. Next: merge `demand` and `inventory` on
`(week_start, part_id)` and compute the derived variables (`available_inventory`,
`units_fulfilled`, `unmet_demand`, `ending_stock`, `stockout`).


## Phase 2: Merge demand + inventory

We merge `demand` and `inventory` on `(week_start, part_id)` to get one row per part per week
with both demand and stock information. Since Phase 1 confirmed both tables have exactly the
same 1872 `(part_id, week_start)` combinations with no duplicates, this merge should produce
exactly 1872 rows — we assert that explicitly to catch any silent fan-out.


In [74]:
# Merge on (week_start, part_id). Using how="left" with demand as the base
# table is a deliberate choice: demand is the table we ultimately care about
# explaining, and Phase 1 already confirmed inventory has a matching row for
# every demand row, so left vs. inner makes no difference here in practice.
df = demand.merge(inventory, on=["week_start", "part_id"], how="left")

# Assert no silent row duplication happened during the merge.
assert len(df) == len(demand), f"Merge changed row count: {len(demand)} -> {len(df)}"
print(f"Merge OK: {len(df)} rows, {df['part_id'].nunique()} parts")

df.head()


Merge OK: 1872 rows, 24 parts


,week_start,part_id,units_demanded,service_center_region,vehicle_type,service_campaign,stock_on_hand,replenishment
0,2024-01-01,P001,3,West,Compact EV,0,25,0
1,2024-01-08,P001,3,West,Compact EV,0,22,0
2,2024-01-15,P001,4,Central,Compact EV,1,19,0
3,2024-01-22,P001,4,North,Compact EV,0,15,30
4,2024-01-29,P001,4,West,Compact EV,0,41,0


### Derived variables

Using the exact formulas specified for this project:

- `available_inventory = stock_on_hand + replenishment` — total units available to fulfill
  demand that week (stock already on hand plus whatever arrived from suppliers).
- `units_fulfilled = min(available_inventory, units_demanded)` — we can't fulfill more than
  either what was demanded or what was available.
- `unmet_demand = max(units_demanded - available_inventory, 0)` — demand left unsatisfied;
  clipped at 0 so a surplus week doesn't produce a negative "unmet" value.
- `ending_stock = max(available_inventory - units_demanded, 0)` — leftover stock after
  fulfilling demand; clipped at 0 for the same reason.
- `stockout = 1 if unmet_demand > 0 else 0` — a simple flag for whether the part ran out
  that week, used later for stockout-rate analysis.


In [ ]:
# Derived variables, computed exactly as specified. We use np.minimum/np.maximum
# (not Python's min/max) because these operate element-wise across a whole column
# at once, rather than on a single pair of values.
df["available_inventory"] = df["stock_on_hand"] + df["replenishment"]
df["units_fulfilled"] = np.minimum(df["available_inventory"], df["units_demanded"])
df["unmet_demand"] = np.maximum(df["units_demanded"] - df["available_inventory"], 0)
df["ending_stock"] = np.maximum(df["available_inventory"] - df["units_demanded"], 0)
df["stockout"] = (df["unmet_demand"] > 0).astype(int)

df[["week_start", "part_id", "units_demanded", "stock_on_hand", "replenishment",
    "available_inventory", "units_fulfilled", "unmet_demand", "ending_stock", "stockout"]].head(10)


### Sanity checks

Before trusting these derived columns for any downstream analysis, we verify three things
hold across the *entire* dataset, not just the few rows we eyeballed above:

1. **Reconciliation**: total `units_demanded` must equal total `units_fulfilled` + total
   `unmet_demand`. This is a direct consequence of the formulas, but computing it end-to-end
   catches any implementation mistake.
2. **No negatives**: none of the five derived columns should ever be negative (by
   construction, thanks to `min`/`max`, but we verify rather than assume).
3. **Chaining property**: for a given part, this week's `ending_stock` should equal next
   week's `stock_on_hand` — that's how the physical inventory system works, and we saw it
   hold by eye for P001 above. We check it holds for every part, every week.


In [ ]:
# Check 1: total demand reconciles with fulfilled + unmet.
total_demanded = df["units_demanded"].sum()
total_fulfilled = df["units_fulfilled"].sum()
total_unmet = df["unmet_demand"].sum()
check1 = total_demanded == total_fulfilled + total_unmet
print(f"[{'PASS' if check1 else 'FAIL'}] total_demanded ({total_demanded}) == "
      f"total_fulfilled ({total_fulfilled}) + total_unmet ({total_unmet})")

# Check 2: no negatives in any derived column.
derived_cols = ["available_inventory", "units_fulfilled", "unmet_demand", "ending_stock"]
n_negative = (df[derived_cols] < 0).sum().sum()
check2 = n_negative == 0
print(f"[{'PASS' if check2 else 'FAIL'}] negative values across derived columns: {n_negative}")

# Check 3: chaining property, per part. Sort by part then week, shift ending_stock
# down by one row within each part group, and compare to that row's stock_on_hand.
df_sorted = df.sort_values(["part_id", "week_start"]).copy()
df_sorted["prev_ending_stock"] = df_sorted.groupby("part_id")["ending_stock"].shift(1)

# The first week of each part has no "previous" row, so we exclude those (NaN) rows.
chain_check = df_sorted.dropna(subset=["prev_ending_stock"])
mismatches = chain_check[chain_check["prev_ending_stock"] != chain_check["stock_on_hand"]]
check3 = len(mismatches) == 0
print(f"[{'PASS' if check3 else 'FAIL'}] chaining property holds for "
      f"{len(chain_check) - len(mismatches)}/{len(chain_check)} part-week pairs")
if len(mismatches) > 0:
    print(mismatches[["part_id", "week_start", "prev_ending_stock", "stock_on_hand"]].head())


### Layer 1 summary — GO

The full reconstruction is verified end-to-end:
- Data audit found no missing values, no duplicate keys, no date gaps, and no negative values
  across all three source tables.
- The merge of `demand` and `inventory` on `(week_start, part_id)` produced exactly the
  expected 1872 rows, with no fan-out.
- The five derived variables reconcile perfectly: total demand (25,034 units) equals total
  fulfilled (24,476) plus total unmet (558) — an overall unmet-demand rate of about 2.2%.
- The physical chaining property (one week's `ending_stock` equals the next week's
  `stock_on_hand`) holds for all 1,848 part-week transitions checked.

`df` is now a trustworthy foundation for Phase 3 (exploratory analysis of demand, stockouts,
fill rate, and cost exposure) and everything that follows.


## Phase 3: Exploratory data analysis

With a clean, verified `df`, we now explore the data itself, building toward two things:
which forecasting methods are even appropriate for which parts (some parts have steady
demand, others have long stretches of zero demand — that changes which method makes sense),
and where the cost exposure actually sits (which parts are expensive to under- or over-stock).

This section covers: demand character per part, a reality check on the synthetic
`demand_trend` labels, demand by category/region, inventory KPIs (stockout rate, fill rate),
historical holding/stockout cost per part, and a cost-vs-volume segmentation. We build up a
single `part_profile` table across these steps that later phases (forecasting, reorder logic)
will reuse.

### Demand character per part

For each part we compute the mean and standard deviation of weekly demand, and the share of
weeks with zero demand, as basic descriptive statistics. The proper classification of demand
*pattern* (smooth vs. intermittent) follows in the next cells using a standard method, rather
than an ad-hoc threshold on zero-demand share alone.


In [ ]:
# Per-part demand statistics: mean, standard deviation, and share of zero-demand weeks.
part_profile = df.groupby("part_id")["units_demanded"].agg(
    mean_demand="mean",
    std_demand="std",
    n_weeks="count",
).reset_index()

zero_share = df.groupby("part_id")["units_demanded"].apply(lambda s: (s == 0).mean())
part_profile["zero_demand_share"] = part_profile["part_id"].map(zero_share)

# Attach part name and category for readability, and carry demand_trend along
# so we can compare the synthetic label against what we actually observe later.
part_profile = part_profile.merge(
    parts[["part_id", "part_name", "category", "demand_trend"]], on="part_id"
)

part_profile.sort_values("zero_demand_share", ascending=False)


### Demand pattern classification (Syntetos-Boylan)

Rather than an arbitrary threshold on zero-demand share, we use the standard method from
demand-planning literature for deciding when intermittent-demand forecasting (Croston's
method) is justified. Two measures, computed per part from its own weekly demand history:

- **ADI (Average Demand Interval)**: the average number of weeks between weeks with nonzero
  demand, computed as `total weeks / number of nonzero-demand weeks`. ADI = 1 means demand
  happens every week; ADI = 5 means demand happens roughly once every 5 weeks on average.
- **CV² (squared coefficient of variation)**: computed only over the nonzero-demand weeks,
  as `(std of nonzero demand / mean of nonzero demand)^2`. Low CV² means that when demand
  does happen, the quantity is fairly consistent; high CV² means it swings wildly.

Standard cutoffs (Syntetos & Boylan): **ADI > 1.32** flags infrequent demand, **CV² > 0.49**
flags erratic order sizes. Combining both gives four categories:

| | CV² <= 0.49 | CV² > 0.49 |
|---|---|---|
| **ADI <= 1.32** | Smooth | Erratic |
| **ADI > 1.32** | Intermittent | Lumpy |

Croston's method (Phase 4) is only justified for parts landing in Intermittent or Lumpy.


In [ ]:
def adi_cv2(units_demanded):
    """Compute ADI and CV^2 for one part's weekly demand series."""
    n_weeks = len(units_demanded)
    nonzero = units_demanded[units_demanded > 0]
    n_nonzero = len(nonzero)

    adi = n_weeks / n_nonzero
    cv2 = (nonzero.std() / nonzero.mean()) ** 2
    return pd.Series({"ADI": adi, "CV2": cv2})

# Apply the function to each part's demand series and get one row per part back.
# unstack() leaves part_id as the index, so reset_index() turns it back into a
# regular column before we merge it into part_profile.
adi_cv2_table = df.groupby("part_id")["units_demanded"].apply(adi_cv2).unstack().reset_index()
part_profile = part_profile.merge(adi_cv2_table, on="part_id")

# Classify using the standard Syntetos-Boylan cutoffs.
def classify(row):
    if row["ADI"] <= 1.32 and row["CV2"] <= 0.49:
        return "Smooth"
    elif row["ADI"] > 1.32 and row["CV2"] <= 0.49:
        return "Intermittent"
    elif row["ADI"] <= 1.32 and row["CV2"] > 0.49:
        return "Erratic"
    else:
        return "Lumpy"

part_profile["demand_pattern"] = part_profile.apply(classify, axis=1)

print(part_profile["demand_pattern"].value_counts())
part_profile.sort_values("ADI", ascending=False)[
    ["part_id", "part_name", "mean_demand", "zero_demand_share", "ADI", "CV2", "demand_pattern"]
]


**Finding**: all 24 parts classify as **Smooth** under the Syntetos-Boylan method — ADI is
approximately 1.0 for every part (demand occurs almost every week, no meaningful gaps) and
CV² stays well under the 0.49 erratic-sizing threshold (ranging roughly 0.05-0.23, i.e. order
sizes are fairly consistent when demand happens). Unlike the earlier ad-hoc 30% rule, this
result comes from a real, citable methodology, so we can state with confidence:
**Croston's method is not justified for any part in this dataset** — standard forecasting
methods (naive, moving average, exponential smoothing, Holt-Winters) are appropriate across
the board. We will not implement Croston's in Phase 4.


### Trend-label reality check

`parts.csv` includes a synthetic `demand_trend` label (Increasing, Decreasing, Stable,
Seasonal) that's meant to describe each part's demand behavior. We haven't verified these
labels against the actual weekly data yet. Since Holt-Winters (Phase 4) explicitly models
trend and seasonality, using it would only be justified for parts whose labels are backed up
by what the data actually shows — so we plot a sample of parts per label and check by eye
whether the label matches the pattern.


In [ ]:
import matplotlib.pyplot as plt

# Fixed categorical colors (not cycled/reassigned across plots) so each part's
# line color always means the same thing within a subplot.
line_colors = ["#2a78d6", "#1baf7a", "#eda100"]  # blue, aqua, yellow

trend_labels = sorted(parts["demand_trend"].unique())
fig, axes = plt.subplots(len(trend_labels), 1, figsize=(10, 3 * len(trend_labels)), sharex=True)

for ax, label in zip(axes, trend_labels):
    # Sample up to 3 parts carrying this label.
    sample_ids = parts.loc[parts["demand_trend"] == label, "part_id"].head(3)
    for color, pid in zip(line_colors, sample_ids):
        series = df[df["part_id"] == pid].sort_values("week_start")
        ax.plot(series["week_start"], series["units_demanded"], color=color, linewidth=2, label=pid)
    ax.set_title(f"demand_trend label: {label}")
    ax.set_ylabel("Units demanded")
    ax.legend(loc="upper right", frameon=False)

axes[-1].set_xlabel("Week")
fig.tight_layout()
plt.show()


**Finding**: the synthetic `demand_trend` labels are only a partial match to the actual data.
"Stable" held up well (all 3 sampled parts look flat, no long-term drift). "Increasing" and
"Decreasing" each matched for 2 of 3 sampled parts. "Seasonal" was not visually confirmable —
with only ~78 weeks (1.5 years) of data, a repeating annual cycle is hard to distinguish from
general volatility by eye alone.

**Decision**: rather than excluding Holt-Winters based on this ambiguous visual check, we will
implement it alongside naive, moving average, and exponential smoothing for every part in
Phase 4, and let the dual evaluation (error vs. cost) show empirically whether the added
trend/seasonality modeling actually helps. This also directly serves the project's central
question — testing whether a more sophisticated method earns its complexity. Croston's method
remains excluded, based on the much stronger ADI/CV² evidence above.


### Demand by category and region

Next we look at total demand grouped two ways: by part `category` (which product families
carry the most volume?) and by `service_center_region` (do regions differ meaningfully in
what they order?). This establishes where the business's demand actually concentrates, which
matters for prioritizing which parts/categories deserve the most attention in the cost
analysis later.


In [ ]:
# Total demand by category: bring in `category` from parts, then sum units_demanded.
category_demand = (
    df.merge(parts[["part_id", "category"]], on="part_id")
    .groupby("category")["units_demanded"].sum()
    .sort_values(ascending=False)
)

# Total demand by region: service_center_region already lives in demand/df directly.
region_demand = (
    df.groupby("service_center_region")["units_demanded"].sum()
    .sort_values(ascending=False)
)

print(category_demand)
print()
print(region_demand)

# A single measure grouped by category doesn't need a multi-color legend -
# the x-axis labels already identify each bar, so one flat color is used.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(category_demand.index, category_demand.values, color="#2a78d6")
axes[0].set_title("Total demand by category")
axes[0].set_ylabel("Total units demanded")
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(region_demand.index, region_demand.values, color="#2a78d6")
axes[1].set_title("Total demand by region")
axes[1].set_ylabel("Total units demanded")

fig.tight_layout()
plt.show()


### Inventory KPIs per part

We now measure how well the inventory system actually performed for each part, not just how
much was demanded. Six KPIs, computed per part over the full 78-week history:

- **stockout_rate**: fraction of weeks the part ran out (`stockout` averaged over all weeks).
- **fill_rate**: fraction of total demand actually fulfilled (`sum(units_fulfilled) / sum(units_demanded)`).
- **total_unmet_demand**: total units of demand never satisfied, summed over all weeks.
- **mean_ending_stock**: average leftover stock at the end of a week.
- **weeks_of_supply**: `mean_ending_stock / mean_demand` — on average, how many weeks of
  demand the leftover stock could cover. A high number suggests excess inventory sitting idle.
- **stockout_severity**: average `unmet_demand`, but only counted over weeks where a stockout
  actually happened (`stockout == 1`). This separates "stocks out often but only misses a
  little" from "rarely stocks out but misses a lot when it does" — two very different risk
  profiles that `stockout_rate` alone can't distinguish. Parts that never stocked out get 0.


In [ ]:
# Core KPIs computed directly via groupby-agg.
kpi = df.groupby("part_id").agg(
    stockout_rate=("stockout", "mean"),
    total_unmet_demand=("unmet_demand", "sum"),
    total_fulfilled=("units_fulfilled", "sum"),
    total_demanded=("units_demanded", "sum"),
    mean_ending_stock=("ending_stock", "mean"),
).reset_index()

kpi["fill_rate"] = kpi["total_fulfilled"] / kpi["total_demanded"]

# Stockout severity: mean unmet_demand, computed only over weeks with an actual
# stockout. Parts with zero stockout weeks won't appear in this groupby result at
# all, so we map onto the full part list and fill missing entries with 0.
severity = df[df["stockout"] == 1].groupby("part_id")["unmet_demand"].mean()
kpi["stockout_severity"] = kpi["part_id"].map(severity).fillna(0)

# weeks_of_supply needs mean_demand, which already lives in part_profile.
kpi = kpi.merge(part_profile[["part_id", "mean_demand"]], on="part_id")
kpi["weeks_of_supply"] = kpi["mean_ending_stock"] / kpi["mean_demand"]

# Fold the new KPIs into part_profile.
part_profile = part_profile.merge(
    kpi[["part_id", "stockout_rate", "fill_rate", "total_unmet_demand",
         "mean_ending_stock", "weeks_of_supply", "stockout_severity"]],
    on="part_id",
)

part_profile.sort_values("stockout_rate", ascending=False)[
    ["part_id", "part_name", "stockout_rate", "fill_rate", "total_unmet_demand",
     "stockout_severity", "mean_ending_stock", "weeks_of_supply"]
]


### Historical cost per part

Now we translate what actually happened (leftover stock, unmet demand) into money, using the
cost parameters from `parts.csv`. **Cost assumption, stated explicitly**: this is the
*historical* cost the business actually incurred given what really happened — not a
forecast-error cost (that comes later in Phase 5, once we have forecasts to compare against
actuals). Two components, summed over all 78 weeks per part:

- **historical_holding_cost** = `sum(ending_stock x holding_cost_per_unit_week)` — the cost of
  whatever stock was sitting around unsold at the end of each week.
- **historical_stockout_cost** = `sum(unmet_demand x stockout_cost_per_unit)` — the cost of
  whatever demand went unfulfilled each week.
- **historical_total_cost** = the two added together.

This tells us, for each part, whether its dominant historical cost driver was carrying too
much stock or missing too much demand — a first look at cost exposure, before we've even
started forecasting.


In [ ]:
# Bring in the two cost-rate parameters from parts.csv, then compute weekly
# holding cost and stockout cost per row before summing per part.
cost_cols = ["part_id", "holding_cost_per_unit_week", "stockout_cost_per_unit"]
df_cost = df.merge(parts[cost_cols], on="part_id")

df_cost["weekly_holding_cost"] = df_cost["ending_stock"] * df_cost["holding_cost_per_unit_week"]
df_cost["weekly_stockout_cost"] = df_cost["unmet_demand"] * df_cost["stockout_cost_per_unit"]

historical_cost = df_cost.groupby("part_id").agg(
    historical_holding_cost=("weekly_holding_cost", "sum"),
    historical_stockout_cost=("weekly_stockout_cost", "sum"),
).reset_index()
historical_cost["historical_total_cost"] = (
    historical_cost["historical_holding_cost"] + historical_cost["historical_stockout_cost"]
)

part_profile = part_profile.merge(historical_cost, on="part_id")

print("=== Top 5 by historical holding cost ===")
print(part_profile.sort_values("historical_holding_cost", ascending=False)
      [["part_id", "part_name", "historical_holding_cost"]].head())

print("\n=== Top 5 by historical stockout cost ===")
print(part_profile.sort_values("historical_stockout_cost", ascending=False)
      [["part_id", "part_name", "historical_stockout_cost"]].head())

print("\n=== Top 5 by historical total cost ===")
part_profile.sort_values("historical_total_cost", ascending=False)[
    ["part_id", "part_name", "historical_holding_cost", "historical_stockout_cost", "historical_total_cost"]
].head()


### Segmentation: demand volume vs. unit cost

We plot each part on two axes — mean weekly demand (volume) and unit cost — with color
showing historical stockout cost exposure. The median of each axis splits the parts into four
quadrants: cheap/low-demand, cheap/high-demand ("cheap fast-movers"), expensive/low-demand
("expensive rarities"), and expensive/high-demand. We use the **median** of each axis as the
dividing line — a data-driven split rather than an arbitrary dollar or volume threshold, since
there's no externally given business number for "expensive" or "high-demand" in this dataset.
This segmentation previews the four reorder-recommendation categories built in Phase 6.


In [ ]:
from matplotlib.colors import LinearSegmentedColormap

profile = part_profile.merge(parts[["part_id", "unit_cost"]], on="part_id")

median_demand = profile["mean_demand"].median()
median_cost = profile["unit_cost"].median()

# Sequential single-hue (blue, light->dark) colormap for the stockout-cost magnitude.
blue_steps = ["#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec", "#5598e7",
              "#3987e5", "#2a78d6", "#256abf", "#1c5cab", "#184f95", "#104281", "#0d366b"]
seq_blue = LinearSegmentedColormap.from_list("seq_blue", blue_steps)

fig, ax = plt.subplots(figsize=(9, 7))
scatter = ax.scatter(
    profile["mean_demand"], profile["unit_cost"],
    c=profile["historical_stockout_cost"], cmap=seq_blue,
    s=110, edgecolor="#0b0b0b", linewidth=0.5,
)

# Quadrant dividers at the median of each axis.
ax.axvline(median_demand, color="#898781", linestyle="--", linewidth=1)
ax.axhline(median_cost, color="#898781", linestyle="--", linewidth=1)

# Label each point with its part_id so individual parts can be identified.
for _, row in profile.iterrows():
    ax.annotate(row["part_id"], (row["mean_demand"], row["unit_cost"]),
                fontsize=8, xytext=(4, 4), textcoords="offset points")

ax.set_xlabel("Mean weekly demand (units)")
ax.set_ylabel("Unit cost ($)")
ax.set_title("Part segmentation: demand volume vs. unit cost\n(color = historical stockout cost)")
cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label("Historical stockout cost ($)")
fig.tight_layout()
plt.show()

# Assign each part to a quadrant using the median splits, then list members.
profile["demand_level"] = np.where(profile["mean_demand"] > median_demand, "high-demand", "low-demand")
profile["cost_level"] = np.where(profile["unit_cost"] > median_cost, "expensive", "cheap")
profile["quadrant"] = profile["cost_level"] + " / " + profile["demand_level"]

for quadrant, group in profile.groupby("quadrant"):
    print(f"\n{quadrant} ({len(group)} parts):")
    print(group[["part_id", "part_name"]].to_string(index=False))


## Phase 3 conclusions

**Demand character**: all 24 parts classify as Smooth under the Syntetos-Boylan test — no
intermittent demand anywhere in this dataset. Croston's method is excluded from Phase 4 on
this basis. The synthetic `demand_trend` labels only partially matched observed behavior
(Stable held up, Increasing/Decreasing were ~2/3 confirmed, Seasonal was unconfirmable by eye)
— Holt-Winters will still be implemented for every part, letting Phase 5's cost evaluation
decide empirically whether its added complexity is worth it.

**Demand concentration**: Cables and Connectors and Thermal Management are the two largest
categories by volume; regions are comparatively even (no single region dominates or is
negligible).

**Inventory performance**: the 24 parts split cleanly into ~11 with real stockout exposure
(stockout rates from 1.3% up to 16.7%) and ~13 with zero stockouts. Even among the zero-
stockout parts, buffer stock varies widely (`weeks_of_supply` from ~1.4 to ~4.5 weeks) —
some parts avoided stockouts with a much larger safety margin than others needed.

**Cost exposure — the key finding**: for the parts with the worst historical cost, stockout
cost dominates holding cost by roughly 10x (e.g. P008: $2,660 holding vs. $31,590 stockout).
This is a real cost asymmetry: under-forecasting is far more expensive than over-forecasting
for these high-demand, high-unit-cost parts. Conversely, P001 (the single most expensive part
by unit cost) shows the opposite pattern — its cost exposure is almost entirely holding cost,
driven by its high price rather than any stockout risk.

**Segmentation**: the demand-volume-vs-unit-cost quadrants (median split) line up with the
cost findings. The *expensive/high-demand* quadrant (P002, P004, P008, P011, P018) carries
the largest stockout cost exposure and should be prioritized for reliable replenishment. The
*expensive/low-demand* quadrant (P001, P006, P010, P012, P013, P014, P020) carries the
largest holding-cost risk and should be watched for overstocking. The two *cheap* quadrants
carry comparatively little cost exposure either way.

**Hypotheses for Phase 4/5**: given the strong stockout/holding cost asymmetry found above, we
expect that for parts in the expensive/high-demand quadrant, a forecasting method with even a
slight under-forecasting bias will generate disproportionately high cost relative to its
statistical error — while for parts like P001 (expensive/low-demand), an over-forecasting
bias should be the costlier direction. This is the mechanism we expect to drive any
disagreement between the accuracy-based and cost-based method rankings in Phase 5, and is a
direct, testable answer to this project's central question.


## Phase 4: Forecasting

### Train/test split

We use a time-based split, applied identically to every part — never shuffled, since
shuffling would let a method "see the future" relative to what it's predicting. With 78 total
weeks, we use a **75/25 split**: the first ~58 weeks for training, the last ~20 weeks held out
as the test set. The same cutoff date is used for all 24 parts, so every part's forecasts are
evaluated over the same calendar weeks.


In [ ]:
# All weeks, in chronological order (already confirmed complete and gap-free in Phase 1).
all_weeks = sorted(df["week_start"].unique())
n_weeks_total = len(all_weeks)

# 75/25 split: the cutoff index marks the first week of the test set.
split_idx = int(n_weeks_total * 0.75)
train_weeks = all_weeks[:split_idx]
test_weeks = all_weeks[split_idx:]
cutoff_date = test_weeks[0]

print(f"Total weeks: {n_weeks_total}")
print(f"Train weeks: {len(train_weeks)} ({train_weeks[0]} to {train_weeks[-1]})")
print(f"Test weeks: {len(test_weeks)} ({test_weeks[0]} to {test_weeks[-1]})")
print(f"Cutoff date (first test week): {cutoff_date}")


### Forecasting functions

Each method below is a pure function: given a part's demand history so far (a pandas Series,
oldest to newest), it returns a single integer forecast for the *next* week. No I/O, no shared
state — this keeps them simple to test individually and to call uniformly from the
walk-forward engine next.

- **naive**: forecast = last observed value. The simplest possible baseline.
- **moving_average(k=4)**: forecast = mean of the last *k* weeks.
- **exp_smoothing(alpha=0.3)**: forecast = a weighted average that favors recent weeks more,
  computed by recursively updating a "level" estimate over the whole history.
- **holt (trend-aware)**: forecast = level + trend extrapolated one step forward, fit via
  statsmodels' `ExponentialSmoothing` with an additive trend and **no seasonal component** —
  a full Holt-Winters seasonal fit needs at least 2 complete seasonal cycles of data (e.g.
  104+ weeks for an annual cycle), and we only have 78 weeks total, so a seasonal term can't
  be reliably estimated here. This is Holt's linear trend method, the direct predecessor to
  Holt-Winters before the seasonal term was added.

All four are rounded to the nearest integer before returning, since forecasts feed into
integer inventory decisions (per the project's rounding requirement).


In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing


def naive_forecast(history):
    """Forecast = the most recent observed value."""
    return int(round(history.iloc[-1]))


def moving_average_forecast(history, k=4):
    """Forecast = mean of the last k observed values."""
    window = history.iloc[-k:]
    return int(round(window.mean()))


def exp_smoothing_forecast(history, alpha=0.3):
    """Forecast = simple exponential smoothing level, updated recursively
    across the whole history: level = alpha * value + (1 - alpha) * level."""
    level = history.iloc[0]
    for value in history.iloc[1:]:
        level = alpha * value + (1 - alpha) * level
    return int(round(level))


def holt_forecast(history):
    """Forecast = level + trend extrapolated one step, via statsmodels'
    ExponentialSmoothing with an additive trend and no seasonal component
    (see markdown above for why seasonal is omitted). Trend extrapolation
    can produce a negative forecast for a strongly declining series, which
    we clip to 0 since demand can't be negative. Uses an explicit if/else
    instead of max() so this can never be broken by a shadowed builtin."""
    model = ExponentialSmoothing(history.values.astype(float), trend="add", seasonal=None,
                                  initialization_method="estimated")
    fit = model.fit()
    forecast_value = float(np.asarray(fit.forecast(1)).reshape(-1)[0])
    if forecast_value < 0:
        forecast_value = 0.0
    return int(round(forecast_value))


# Quick smoke test on one part's training history, to confirm all four
# functions run and produce sane (non-negative integer) output.
sample_history = df.loc[(df["part_id"] == "P001") & (df["week_start"].isin(train_weeks)),
                         "units_demanded"].reset_index(drop=True)

print("Sample history (last 8 weeks):", sample_history.tail(8).tolist())
print("naive:", naive_forecast(sample_history))
print("moving_average:", moving_average_forecast(sample_history))
print("exp_smoothing:", exp_smoothing_forecast(sample_history))
print("holt:", holt_forecast(sample_history))


### Walk-forward engine

This is the core forecasting loop: for every part, for every test week, for every method, we
forecast that week using **only** the data strictly before it — never anything from that week
or later. As we move forward through the test weeks, each week's actual demand becomes
available and gets folded into the history for forecasting the *next* week (this is why it's
called "walk-forward": the forecast origin walks forward one week at a time, exactly mimicking
how a real planner would operate, never peeking at the future).

Note this means the history used grows over the test period — it always starts at the very
first training week and extends up to (but not including) the week being forecast, so by the
last test week the history includes all 58 training weeks plus the 19 test weeks that have
already "happened."

The output is a single long table: one row per (part, week, method) combination, with the
forecast and the actual value side by side, ready for both error and cost evaluation in
Phase 5.


In [ ]:
methods = {
    "naive": naive_forecast,
    "moving_average": moving_average_forecast,
    "exp_smoothing": exp_smoothing_forecast,
    "holt": holt_forecast,
}

results = []

# One part at a time: get that part's full demand series indexed by week,
# sorted chronologically (required for "history so far" slicing to make sense).
for part_id, part_rows in df.sort_values("week_start").groupby("part_id"):
    part_series = part_rows.set_index("week_start")["units_demanded"]

    for test_week in test_weeks:
        # History = every week strictly before this one - never the test week
        # itself or anything after it.
        history = part_series.loc[part_series.index < test_week]
        actual = int(part_series.loc[test_week])

        for method_name, method_fn in methods.items():
            forecast = method_fn(history)
            results.append({
                "part_id": part_id,
                "week_start": test_week,
                "method": method_name,
                "forecast": forecast,
                "actual": actual,
            })

results_df = pd.DataFrame(results)
print(f"Total forecast rows: {len(results_df)} "
      f"({df['part_id'].nunique()} parts x {len(test_weeks)} test weeks x {len(methods)} methods)")
results_df.head(12)


### Leakage & alignment checks

Before trusting `results_df` for evaluation, we verify the walk-forward loop behaved exactly
as intended — no data leakage, no misalignment, and clean output types:

1. Every `(part_id, method)` combination has exactly 20 rows (one per test week) — confirms
   no test week was skipped or duplicated for any part/method.
2. Every forecasted week actually falls in the test set, and none fall in the training set —
   confirms we never accidentally forecasted a week we'd already trained on.
3. All forecasts are non-negative integers — confirms the rounding/clipping logic in each
   forecasting function worked as intended across all 1920 rows, not just the smoke-test case.
4. No missing values anywhere in the results table.

**Note on the ConvergenceWarnings**: the Holt method occasionally failed to fully converge
during optimization for some of the 480 individual fits (one fit per part per test week). This
is expected with short series (58-77 points) and doesn't break anything — statsmodels still
returns its best available parameter estimates — but it's a real limitation of fitting Holt's
method fresh at every single walk-forward step on this little data, worth keeping in mind when
interpreting Holt's results in Phase 5.


In [ ]:
# Check 1: every (part, method) combination has exactly the same 20 test weeks.
counts = results_df.groupby(["part_id", "method"]).size()
check1 = (counts == len(test_weeks)).all()
print(f"[{'PASS' if check1 else 'FAIL'}] every (part, method) has {len(test_weeks)} rows: {check1}")

# Check 2: every forecasted week is actually in the test set, never in train.
check2 = results_df["week_start"].isin(test_weeks).all() and not results_df["week_start"].isin(train_weeks).any()
print(f"[{'PASS' if check2 else 'FAIL'}] all forecast weeks fall in the test set, none in train: {check2}")

# Check 3: all forecasts are non-negative integers.
all_int = results_df["forecast"].apply(lambda x: isinstance(x, (int, np.integer))).all()
all_nonneg = (results_df["forecast"] >= 0).all()
print(f"[{'PASS' if all_int else 'FAIL'}] all forecasts are integers: {all_int}")
print(f"[{'PASS' if all_nonneg else 'FAIL'}] all forecasts are non-negative: {all_nonneg}")

# Check 4: no missing values anywhere in the results table.
check4 = results_df.isna().sum().sum() == 0
print(f"[{'PASS' if check4 else 'FAIL'}] no NaNs in results_df: {check4}")


### Phase 4 summary — GO

All leakage/alignment checks passed: 1920 forecast rows (24 parts x 20 test weeks x 4
methods), every combination complete, every forecast a non-negative integer, no NaNs, and no
test-week forecasts leaking training data. `results_df` (part_id, week_start, method, forecast,
actual) is ready for Phase 5 — computing statistical error (MSE, MAE, MAPE) and business cost
(overage/underage) per method, and directly answering the central question.


## Phase 5: Dual evaluation

### Error metrics

For each `(part_id, method)` combination, we compute the three statistical error metrics
named in the project brief, plus RMSE for free (it's just `sqrt(MSE)`, same information on
the natural scale):

- **MSE**: mean squared error — penalizes large errors disproportionately.
- **RMSE**: square root of MSE, back in units/week for interpretability.
- **MAE**: mean absolute error — the average size of a miss, in the same units as demand.
- **MAPE**: mean absolute percentage error — the average miss as a percentage of actual
  demand. Zero-demand weeks are excluded (required, to avoid division by zero).

We then average each metric across all 24 parts, per method, to get a first read on which
method is most accurate overall.


In [ ]:
def compute_metrics(group):
    error = group["forecast"] - group["actual"]
    mse = (error ** 2).mean()
    mae = error.abs().mean()

    # MAPE excludes zero-demand weeks to avoid division by zero.
    nonzero = group[group["actual"] > 0]
    mape = (nonzero["forecast"] - nonzero["actual"]).abs().div(nonzero["actual"]).mean() * 100

    return pd.Series({"MSE": mse, "RMSE": mse ** 0.5, "MAE": mae, "MAPE": mape})

error_scores = (
    results_df.groupby(["part_id", "method"])
    .apply(compute_metrics)
    .reset_index()
)

print("=== Error scores by method, averaged across all parts ===")
print(error_scores.groupby("method")[["MSE", "RMSE", "MAE", "MAPE"]].mean().round(2))

error_scores.head(10)


**Finding**: naive is clearly the worst method on every metric (MSE 31.57, RMSE 5.23,
MAE 4.19, MAPE 43.5%). Holt is the most accurate overall (MSE 17.59, RMSE 3.88, MAE 3.17,
MAPE 33.66%), narrowly ahead of exponential smoothing and moving average. So by pure
statistical accuracy, the ranking (best to worst) is: **Holt < exp_smoothing < moving_average
< naive**. This sets up the central test: does the cost-based ranking agree, or diverge?

### Cost metrics

**Cost assumption, stated explicitly**: each week's `forecast` is treated as the stocking
decision that would have been made. If `forecast > actual`, the excess is overage (holding
cost); if `forecast < actual`, the shortfall is underage (stockout cost) — the same logic as
the historical cost calculation in Phase 3, now applied to forecast error instead of the
actual historical stock/replenishment numbers:

- `overage = max(forecast - actual, 0)`, `overage_cost = overage x holding_cost_per_unit_week`
- `underage = max(actual - forecast, 0)`, `underage_cost = underage x stockout_cost_per_unit`
- `total_cost = overage_cost + underage_cost`, summed per part per method across all 20 test
  weeks.


In [ ]:
cost_cols = ["part_id", "holding_cost_per_unit_week", "stockout_cost_per_unit"]
results_costed = results_df.merge(parts[cost_cols], on="part_id")

results_costed["overage"] = np.maximum(results_costed["forecast"] - results_costed["actual"], 0)
results_costed["underage"] = np.maximum(results_costed["actual"] - results_costed["forecast"], 0)
results_costed["overage_cost"] = results_costed["overage"] * results_costed["holding_cost_per_unit_week"]
results_costed["underage_cost"] = results_costed["underage"] * results_costed["stockout_cost_per_unit"]
results_costed["total_cost"] = results_costed["overage_cost"] + results_costed["underage_cost"]

cost_scores = results_costed.groupby(["part_id", "method"]).agg(
    total_overage_cost=("overage_cost", "sum"),
    total_underage_cost=("underage_cost", "sum"),
    total_cost=("total_cost", "sum"),
).reset_index()

print("=== Cost scores by method, averaged across all parts ===")
print(cost_scores.groupby("method")[["total_overage_cost", "total_underage_cost", "total_cost"]].mean().round(2))

cost_scores.head(10)


**Finding**: aggregated across all 24 parts, the cost ranking matches the error ranking
exactly — **Holt cheapest ($5,713.59 average total cost) < exp_smoothing < moving_average <
naive most expensive ($8,307.97)**. At the aggregate level there's no disagreement: the most
accurate method is also the cheapest, on average. But averages can hide part-level
disagreement, which is what we check next.

### The two leaderboards (per part)

Averages can mask disagreement at the level of an individual part. For each part, we rank the
four methods separately by MAE (accuracy) and by total cost, and check whether the same
method wins both.


In [ ]:
# Combine error and cost scores side by side.
comparison = error_scores.merge(cost_scores, on=["part_id", "method"])

# Rank methods within each part: rank 1 = best (lowest MAE / lowest total_cost).
comparison["accuracy_rank"] = comparison.groupby("part_id")["MAE"].rank(method="min")
comparison["cost_rank"] = comparison.groupby("part_id")["total_cost"].rank(method="min")

# For each part, which method is most accurate, and which is cheapest?
best_by_accuracy = comparison.loc[comparison["accuracy_rank"] == 1, ["part_id", "method"]] \
    .rename(columns={"method": "most_accurate_method"})
best_by_cost = comparison.loc[comparison["cost_rank"] == 1, ["part_id", "method"]] \
    .rename(columns={"method": "cheapest_method"})

part_leaderboard = best_by_accuracy.merge(best_by_cost, on="part_id")
part_leaderboard["disagree"] = part_leaderboard["most_accurate_method"] != part_leaderboard["cheapest_method"]

n_disagree = part_leaderboard["disagree"].sum()
print(f"Parts where the most-accurate method != the cheapest method: {n_disagree} / {len(part_leaderboard)}")
part_leaderboard


**Finding**: 14 of 27 rows show disagreement (27 rather than 24 because three parts — P005,
P010, P022 — have exact ties for rank 1 on one side, producing an extra row per tie; this is
a genuine tie in the data, not a bug, since integer forecasts on small demand numbers can
easily coincide). Substantially more than half of the parts show the most-accurate method
differing from the cheapest one — a real breakdown of the aggregate agreement seen above.

### Disagreement analysis — quantifying the cost of choosing "accurate" over "cheap"

Knowing *that* 14 parts disagree isn't enough — we need to know *how much* that disagreement
costs, in dollars, to answer the central question concretely.


In [ ]:
# CORRECTED disagreement logic. The earlier approach used
# drop_duplicates(keep="first") to resolve ties for rank-1 accuracy/cost, which
# is order-dependent: for a genuine 2-way tie (P022 ties on accuracy between
# exp_smoothing and moving_average), keep="first" vs keep="last" gives a
# different disagree/agree answer for the exact same data.
#
# The correct, order-independent definition: a part only "disagrees" if NO
# method tied for most-accurate is ALSO tied for cheapest - i.e. even in the
# best case, a planner picking by accuracy could not have landed on the
# cheapest method. This settles P022 as "agree" (moving_average is tied for
# both), which keep="first" had gotten wrong.
rows = []
for part_id, grp in comparison.groupby("part_id"):
    acc_min = grp["MAE"].min()
    accurate_methods = sorted(grp.loc[grp["MAE"] == acc_min, "method"])
    cost_min = grp["total_cost"].min()
    cheap_methods = sorted(grp.loc[grp["total_cost"] == cost_min, "method"])
    disagree = set(accurate_methods).isdisjoint(cheap_methods)

    # Best-case cost among tied accuracy winners (most charitable to a planner
    # picking by accuracy); the alphabetically-first name is shown for
    # readability when there's a tie - this doesn't affect the dollar figures,
    # which are always the true min/tied cost regardless of which name is shown.
    accuracy_winner_cost = grp.loc[grp["method"].isin(accurate_methods), "total_cost"].min()
    rows.append({
        "part_id": part_id,
        "most_accurate_method": accurate_methods[0],
        "cheapest_method": cheap_methods[0],
        "accuracy_winner_cost": accuracy_winner_cost,
        "cheapest_cost": cost_min,
        "disagree": disagree,
    })

impact = pd.DataFrame(rows)
impact["extra_cost"] = impact["accuracy_winner_cost"] - impact["cheapest_cost"]
impact["pct_more_expensive"] = (impact["extra_cost"] / impact["cheapest_cost"]) * 100

disagreeing = impact[impact["disagree"]].sort_values("pct_more_expensive", ascending=False)
print(disagreeing[["part_id", "most_accurate_method", "cheapest_method",
                    "accuracy_winner_cost", "cheapest_cost", "pct_more_expensive"]].round(1))

avg_premium = disagreeing["pct_more_expensive"].mean()
print(f"\nAcross {len(disagreeing)} disagreeing parts, picking the 'most accurate' method "
      f"instead of the cheapest costs {avg_premium:.1f}% more on average.")


**Finding -- the headline result**: across the 11 disagreeing parts, picking the "most
accurate" method instead of the cheapest one costs **35.8% more on average**, ranging from a
modest 5.2% (P005) up to a striking **152.9% more expensive** (P023, where moving_average
looked best on paper but Holt was actually less than half the cost). Two parts stand out for
the report: **P004 and P008** -- both in the *expensive/high-demand* quadrant flagged in
Phase 3 as carrying the highest stockout-cost exposure -- where Holt was the most accurate
method but not the cheapest, confirming the Phase 3 hypothesis directly.

**Mechanism check**: to explain *why* this happens, not just *that* it happens, we check the
bias direction (mean signed forecast error, not absolute) for the accuracy-winner vs.
cost-winner on each disagreeing part. A method biased toward under-forecasting is punished
hard on parts where stockout cost dominates; a method biased toward over-forecasting is
punished hard where holding cost dominates.


In [ ]:
# Bias direction: mean signed error (not absolute) per part+method.
# Positive = tends to over-forecast (overage/holding-cost risk).
# Negative = tends to under-forecast (underage/stockout-cost risk).
results_df["signed_error"] = results_df["forecast"] - results_df["actual"]
bias = results_df.groupby(["part_id", "method"])["signed_error"].mean().reset_index()
bias.columns = ["part_id", "method", "mean_bias"]

# Attach bias for both the accuracy-winner and the cost-winner, for the disagreeing parts.
bias_check = disagreeing.merge(
    bias.rename(columns={"method": "most_accurate_method", "mean_bias": "accuracy_winner_bias"}),
    on=["part_id", "most_accurate_method"]
).merge(
    bias.rename(columns={"method": "cheapest_method", "mean_bias": "cheapest_bias"}),
    on=["part_id", "cheapest_method"]
)

print(bias_check[["part_id", "most_accurate_method", "accuracy_winner_bias",
                   "cheapest_method", "cheapest_bias"]].round(2))


**Finding**: P004 is the clearest confirmation of the mechanism — Holt under-forecasts by
-2.05 units on average, more severely than exp_smoothing's -1.10. Since P004 sits in the
expensive/high-demand quadrant where stockout cost dominates, that extra under-forecasting
bias gets punished heavily by the high `stockout_cost_per_unit`, explaining why Holt is "more
accurate" (lower MAE) yet costlier here. For other parts the picture is a mix of overage and
underage weeks that a single mean-bias number doesn't fully capture — a visual makes this
clearer than the tables alone.

One killer visual: total cost by method, for the parts with the largest accuracy/cost
disagreement.


In [ ]:
# Top 6 disagreement parts, by how much more expensive the "accurate" choice was.
top_disagreement_parts = disagreeing.sort_values("pct_more_expensive", ascending=False)["part_id"].head(6).tolist()

plot_data = comparison[comparison["part_id"].isin(top_disagreement_parts)].copy()
plot_data["part_id"] = pd.Categorical(plot_data["part_id"], categories=top_disagreement_parts, ordered=True)
plot_data = plot_data.sort_values("part_id")

method_order = ["naive", "moving_average", "exp_smoothing", "holt"]
method_colors = {"naive": "#2a78d6", "moving_average": "#1baf7a", "exp_smoothing": "#eda100", "holt": "#008300"}

fig, ax = plt.subplots(figsize=(11, 6))
bar_width = 0.2
x = np.arange(len(top_disagreement_parts))

for i, method in enumerate(method_order):
    method_costs = plot_data[plot_data["method"] == method].set_index("part_id")["total_cost"]
    method_costs = method_costs.reindex(top_disagreement_parts)
    ax.bar(x + i * bar_width, method_costs.values, width=bar_width,
           label=method, color=method_colors[method])

ax.set_xticks(x + bar_width * 1.5)
ax.set_xticklabels(top_disagreement_parts)
ax.set_xlabel("Part")
ax.set_ylabel("Total cost ($)")
ax.set_title("Total cost by method - parts with the largest accuracy/cost disagreement")
ax.legend(title="Method", frameon=False)
fig.tight_layout()
plt.show()


### Robustness check: does the "most accurate" definition matter?

Everything above defines "most accurate" as lowest MAE. Before treating the disagreement count
as solid, we check whether picking a different, equally reasonable accuracy metric (RMSE)
tells a materially different story.

In [ ]:
# Recompute the same disagreement logic, but using RMSE instead of MAE
# to define "most accurate", to check the finding isn't an artifact of one metric.
rmse_rows = []
for part_id, grp in comparison.groupby("part_id"):
    acc_min = grp["RMSE"].min()
    accurate_methods = sorted(grp.loc[grp["RMSE"] == acc_min, "method"])
    cost_min = grp["total_cost"].min()
    cheap_methods = sorted(grp.loc[grp["total_cost"] == cost_min, "method"])
    disagree = set(accurate_methods).isdisjoint(cheap_methods)
    rmse_rows.append({"part_id": part_id, "disagree": disagree})

rmse_disagreement = pd.DataFrame(rmse_rows)
rmse_disagree_parts = set(rmse_disagreement.loc[rmse_disagreement["disagree"], "part_id"])
mae_disagree_parts = set(disagreeing["part_id"])

print(f"MAE-based disagreement count: {len(mae_disagree_parts)} / 24")
print(f"RMSE-based disagreement count: {len(rmse_disagree_parts)} / 24")
print(f"Parts flagged by both metrics: {len(mae_disagree_parts & rmse_disagree_parts)}")
print(f"Only flagged under MAE: {sorted(mae_disagree_parts - rmse_disagree_parts)}")
print(f"Only flagged under RMSE: {sorted(rmse_disagree_parts - mae_disagree_parts)}")

### Central question answered

**Can a forecasting method with low statistical error still create high inventory cost?
Yes.**

Aggregated across all 24 parts, accuracy and cost rankings agreed: Holt was both the most
accurate method (lowest MSE/MAE/MAPE) and the cheapest on average. But that aggregate
agreement broke down at the individual-part level. On 11 of the 24 parts, the most
statistically accurate method was *not* the cheapest one -- and choosing accuracy over cost on
those parts would have cost **35.8% more on average**, peaking at **152.9% more** on a single
part (P023).

The mechanism is cost asymmetry combined with bias direction, not raw error size. On parts
like P004 and P008 -- in the expensive/high-demand quadrant identified in Phase 3, where
stockout cost dominates holding cost by roughly 10x -- Holt's slightly stronger
under-forecasting bias was punished disproportionately by the high per-unit stockout cost,
even though its average error (MAE) was smaller than the alternative. A method's statistical
accuracy says nothing about which *direction* its errors lean, and that direction is what
actually determines cost when overage and underage are priced so differently.

**Robustness check**: is this an artifact of using MAE specifically to define "most accurate"?
Re-running the same disagreement logic with RMSE instead of MAE gives 9 of 24 parts
disagreeing (vs. 11 with MAE) -- a real difference, not identical, but the same direction and
similar magnitude. Only 7 parts are flagged by both metrics. This tells us two things: the
core finding is not a fragile artifact of one specific metric (a large, similarly-sized
disagreement shows up either way), but the *exact* count and which parts are flagged does
depend on which accuracy metric is chosen -- itself a small illustration of the paper's
broader point, that "accuracy" is not one single, unambiguous thing.

**Practical takeaway**: method selection for inventory forecasting cannot rely on error
metrics alone. The same method that is cheapest for most parts (Holt, in this dataset) can be
the most expensive choice on specific parts whose cost structure punishes its particular bias
direction -- meaning per-part cost evaluation, not a single global "best method," is necessary
for genuinely cost-minimizing forecasting decisions.


## Phase 6: Reorder logic

With the dual evaluation complete, we now turn the analysis into concrete action: for each
part, pick its cost-cheapest forecasting method (not necessarily the most accurate one --
that's the whole point of the last section), forecast forward-looking weekly demand, and
compute a safety stock, reorder point, and recommended order quantity, ending in a
four-category recommendation and the exported CSV deliverable.

In [ ]:
# Pick the cost-cheapest method per part. idxmin() picks the first row on an exact
# tie, giving one deterministic choice per part (matters for P005/P010/P022 which
# had ties earlier).
best_method_per_part = comparison.loc[
    comparison.groupby("part_id")["total_cost"].idxmin(), ["part_id", "method"]
].reset_index(drop=True)
best_method_per_part.columns = ["part_id", "selected_method"]

print(best_method_per_part)

# Now forecast one week ahead per part, using its selected method on the FULL
# demand history (not just the training weeks) - this is the actual forward-
# looking forecast for the reorder decision, not a backtest.
forecasted_rows = []
for _, row in best_method_per_part.iterrows():
    pid = row["part_id"]
    method_name = row["selected_method"]
    full_history = df.loc[df["part_id"] == pid].sort_values("week_start")["units_demanded"].reset_index(drop=True)
    forecast_value = methods[method_name](full_history)
    forecasted_rows.append({
        "part_id": pid,
        "selected_method": method_name,
        "forecasted_weekly_demand": forecast_value,
    })

forecasted_demand_df = pd.DataFrame(forecasted_rows)
forecasted_demand_df


### Safety stock and reorder point

**Service level assumption**: we use a fixed 95% service level ($z=1.65$) applied uniformly
to every part. A per-part newsvendor critical ratio (service level derived from each part's
own stockout-cost-to-holding-cost ratio) was tried first, but rejected: `stockout_cost_per_unit`
exceeds `holding_cost_per_unit_week` by 1-2 orders of magnitude for essentially every part in
this dataset, so the ratio saturates near 1.0 regardless of part, losing all differentiating
power. A fixed, standard service level is simpler and equally honest here.

**Two caveats on the demand-variability input, worth stating explicitly**:
- `std_demand` is computed over the *entire* 78-week history, including the test period --
  reasonable for a forward-looking deployment recommendation, but worth noting since it's not
  purely an "in-sample" number.
- For parts with a real trend (Increasing/Decreasing), the raw standard deviation partly
  reflects the trend itself, not just week-to-week noise -- since a steadily rising series has
  higher variance than a flat one at the same noise level. This means safety stock is mildly
  *conservative* (slightly oversized) for trending parts, which errs safe rather than risky,
  but is a simplification worth flagging rather than presenting as exact.

In [ ]:
Z_SERVICE_LEVEL = 1.65  # 95% service level, applied uniformly to all parts

reorder_base = forecasted_demand_df.merge(
    parts[["part_id", "part_name", "category", "lead_time_days"]],
    on="part_id"
).merge(
    part_profile[["part_id", "std_demand"]], on="part_id"
)

reorder_base["lead_time_weeks"] = reorder_base["lead_time_days"] / 7

# Safety stock = z x demand_std x sqrt(lead_time_weeks), rounded UP.
reorder_base["safety_stock"] = np.ceil(
    Z_SERVICE_LEVEL * reorder_base["std_demand"] * np.sqrt(reorder_base["lead_time_weeks"])
).astype(int)

# Reorder point = forecasted demand needed to cover the lead time, plus safety stock.
reorder_base["reorder_point"] = np.ceil(
    reorder_base["forecasted_weekly_demand"] * reorder_base["lead_time_weeks"] + reorder_base["safety_stock"]
).astype(int)

reorder_base[["part_id", "part_name", "lead_time_days", "lead_time_weeks",
              "std_demand", "safety_stock", "reorder_point"]].round(2)


### Recommended order quantity

**Simplification, stated explicitly**: `recommended_order_quantity = max(reorder_point -
current_stock, 0)` restores stock exactly back up to the reorder point. In a stricter
continuous-review system this means the part could trigger another reorder almost
immediately, since the reorder point is (by construction) the *minimum* comfortable level, not
a target buffer above it. Standard practice often orders up to a higher target (e.g. reorder
point plus expected demand over a review period, or a fixed lot size). We keep the simpler
order-up-to-reorder-point rule since the brief asks only for a recommended quantity, not a full
periodic-review policy -- but the fuller version is a natural next iteration.

In [ ]:
# Current stock = ending_stock in the most recent week, per part.
latest_week = df["week_start"].max()
current_stock_df = df.loc[df["week_start"] == latest_week, ["part_id", "ending_stock"]] \
    .rename(columns={"ending_stock": "current_stock"})

reorder_base = reorder_base.merge(current_stock_df, on="part_id")

# Order enough to bring stock back up to the reorder point; never a negative order.
reorder_base["recommended_order_quantity"] = np.maximum(
    reorder_base["reorder_point"] - reorder_base["current_stock"], 0
).astype(int)

reorder_base[["part_id", "part_name", "current_stock", "reorder_point", "recommended_order_quantity"]]


### Recommendation classification

The four categories below are checked **in this order, first match wins** -- i.e. the order
is itself a priority scheme, not an arbitrary listing. An active shortage (needing
replenishment right now) outranks a longer-term risk flag, which outranks the two
cost-avoidance categories (expensive-low-demand, excessive inventory), reflecting that
correcting an active shortage is more time-critical than avoiding future overstock.

In [ ]:
def classify_part(row):
    if row["current_stock"] < row["reorder_point"]:
        return "Needs immediate replenishment"
    elif row["stockout_rate"] > 0.05:
        return "High stockout risk"
    elif row["unit_cost"] > median_cost and row["mean_demand"] < median_demand:
        return "Expensive low-demand part - do not overstock"
    elif row["current_stock"] > 2 * row["reorder_point"]:
        return "Excessive inventory"
    else:
        return "Adequate - no action needed"

# Bring in stockout_rate (Phase 3), unit_cost (parts.csv), and mean_demand (Phase 3)
reorder_base = reorder_base.merge(
    part_profile[["part_id", "stockout_rate", "mean_demand"]], on="part_id"
).merge(
    parts[["part_id", "unit_cost"]], on="part_id"
)

reorder_base["recommendation"] = reorder_base.apply(classify_part, axis=1)

print(reorder_base["recommendation"].value_counts())
reorder_base[["part_id", "part_name", "current_stock", "reorder_point", "recommendation"]]


In [ ]:
import os
os.makedirs("../outputs", exist_ok=True)

output_cols = ["part_id", "part_name", "category", "current_stock", "forecasted_weekly_demand",
               "lead_time_days", "safety_stock", "reorder_point", "recommended_order_quantity", "recommendation"]

final_output = reorder_base[output_cols].copy()

# Validate before writing: all 24 parts present, no missing values, and every
# quantity column is a proper integer (per the project's rounding requirement).
assert final_output["part_id"].nunique() == 24, "Missing parts in final output"
assert final_output.isna().sum().sum() == 0, "NaNs present in final output"
for col in ["current_stock", "forecasted_weekly_demand", "lead_time_days",
            "safety_stock", "reorder_point", "recommended_order_quantity"]:
    assert pd.api.types.is_integer_dtype(final_output[col]), f"{col} is not integer dtype"

final_output.to_csv("../outputs/reorder_recommendations.csv", index=False)
print(f"Saved outputs/reorder_recommendations.csv - {len(final_output)} rows, all checks passed")
final_output


## Managerial summary

**Bottom line: yes, a statistically accurate forecasting method can still be expensive.** Across our 24 spare parts, the forecasting method with the best average accuracy (Holt's trend-aware method) was also the cheapest choice on average -- but on 11 of 24 parts individually, picking the "most accurate" method instead of the actual cheapest one would have cost 35.8% more on average, and on one part, 152.9% more. Accuracy and cost do not automatically agree, part by part.

**Why**: it comes down to which *direction* a method's errors lean, not how big they are on average. Parts where running out is far more costly than overstocking (most of our parts, since stockout cost is 10x+ holding cost almost everywhere) get punished hard by a method that even slightly under-forecasts, regardless of how "accurate" it looks on paper.

**Top actions right now**:
1. **15 of 24 parts need immediate reordering** based on current stock vs. reorder point (see `reorder_recommendations.csv`) -- several show zero stock on hand (P002, P006, P016). This high count is itself a finding, not a red flag in the analysis: the reorder point is deliberately built to a 95% service level, and the historical stock levels (which were never managed against this policy) were clearly run leaner than that -- consistent with the 2.2% historical unmet-demand rate found in Phase 2. In plain terms, the warehouse has been under-buffered relative to the service level this analysis recommends.
2. **P012 is a stockout-risk part to watch** -- stock currently looks adequate, but it has a history of running out (5.1% of weeks) even when not flagged as urgent today.
3. **P001 and P010 are expensive, low-demand parts** -- don't over-order these on a routine restock; their risk is tying up cash in inventory that barely turns over, not running out.
4. **For future forecasting method selection, evaluate cost per part, not one global "best" method** -- Holt worked well overall but was the worst choice specifically on P004 and P008, two of the highest stockout-cost parts in the portfolio.

**What to monitor going forward**: re-run this evaluation periodically as new weekly data arrives -- cost rankings can shift as demand patterns evolve, and a method that's cheap today isn't guaranteed to stay cheap.


In [ ]:
import os
os.makedirs("../report/figures", exist_ok=True)

# Re-save the segmentation scatter (justifies the reorder categories)
fig, ax = plt.subplots(figsize=(7, 5.5))
scatter = ax.scatter(
    profile["mean_demand"], profile["unit_cost"],
    c=profile["historical_stockout_cost"], cmap=seq_blue,
    s=90, edgecolor="#0b0b0b", linewidth=0.5,
)
ax.axvline(median_demand, color="#898781", linestyle="--", linewidth=1)
ax.axhline(median_cost, color="#898781", linestyle="--", linewidth=1)
for _, row in profile.iterrows():
    ax.annotate(row["part_id"], (row["mean_demand"], row["unit_cost"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("Mean weekly demand (units)")
ax.set_ylabel("Unit cost ($)")
ax.set_title("Part segmentation: demand volume vs. unit cost")
cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label("Historical stockout cost ($)")
fig.tight_layout()
fig.savefig("../report/figures/segmentation.png", dpi=200)
plt.show()

# Re-save the disagreement cost-bar chart (the central-question proof)
fig, ax = plt.subplots(figsize=(7.5, 4.5))
bar_width = 0.2
x = np.arange(len(top_disagreement_parts))
for i, method in enumerate(method_order):
    method_costs = plot_data[plot_data["method"] == method].set_index("part_id")["total_cost"].reindex(top_disagreement_parts)
    ax.bar(x + i * bar_width, method_costs.values, width=bar_width, label=method, color=method_colors[method])
ax.set_xticks(x + bar_width * 1.5)
ax.set_xticklabels(top_disagreement_parts)
ax.set_xlabel("Part")
ax.set_ylabel("Total cost ($)")
ax.set_title("Total cost by method - largest accuracy/cost disagreements")
ax.legend(title="Method", frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig("../report/figures/disagreement.png", dpi=200)
plt.show()

print("Saved figures to ../report/figures/")
